# Analise de divergencia de cotacoes e comparacao da carteira vencedora

Este notebook tem dois objetivos:

1. Sinalizar cotacoes suspeitas entre o historico da carteira e o `fundamentus_data.parquet`.
2. Recalcular a comparacao ativo a ativo da carteira vencedora usando as cotacoes do `fundamentus_data.parquet`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

BASE_DIR = Path("..").resolve()
ANALYSIS_PATH = BASE_DIR / "data/performance_period_analysis.parquet"
HISTORY_PATH = BASE_DIR / "data/performance_committed_since_2026-03-10.parquet"
FUND_PATH = BASE_DIR / "data/fundamentus_data.parquet"

In [2]:
analysis = pd.read_parquet(ANALYSIS_PATH)
history = pd.read_parquet(HISTORY_PATH)
fund = pd.read_parquet(FUND_PATH)

history["Data"] = pd.to_datetime(history["Data"])
history["commit_committed_at"] = pd.to_datetime(history["commit_committed_at"], utc=True)

fund = fund.reset_index(names="papel")
fund["update date"] = pd.to_datetime(fund["update date"])

latest_history = (
    history.sort_values("commit_committed_at")
    .drop_duplicates(
        subset=["Estratégia", "Volume Mínimo", "Ativos na Carteira", "Data", "papel"],
        keep="last",
    )
    .reset_index(drop=True)
)

latest_fund = (
    fund.sort_values(["papel", "update date"])
    .drop_duplicates(subset=["papel", "update date"], keep="last")
    .reset_index(drop=True)
)

print(f"analysis rows: {len(analysis)}")
print(f"latest history rows: {len(latest_history)}")
print(f"latest fund rows: {len(latest_fund)}")

analysis rows: 72
latest history rows: 2730
latest fund rows: 6956


## 1. Divergencias suspeitas entre fontes

A regra abaixo compara a cotacao usada no historico da carteira com a cotacao do `fundamentus_data.parquet` para o mesmo ticker e a mesma data.

A divergencia e marcada como suspeita quando a diferenca relativa absoluta e maior ou igual a `20%`.

In [3]:
SUSPECT_THRESHOLD = 0.20

quote_compare = (
    latest_history[
        ["papel", "Data", "Estratégia", "Volume Mínimo", "Ativos na Carteira", "Cotação", "Quantidade", "Valor"]
    ]
    .rename(columns={"Cotação": "cotacao_carteira"})
    .merge(
        latest_fund[["papel", "update date", "Cotação"]].rename(
            columns={"update date": "Data", "Cotação": "cotacao_fundamentus"}
        ),
        on=["papel", "Data"],
        how="left",
    )
)

quote_compare["dif_abs"] = quote_compare["cotacao_carteira"] - quote_compare["cotacao_fundamentus"]
quote_compare["dif_rel"] = np.where(
    quote_compare["cotacao_fundamentus"].notna() & quote_compare["cotacao_fundamentus"].ne(0),
    quote_compare["dif_abs"] / quote_compare["cotacao_fundamentus"],
    np.nan,
)
quote_compare["suspeita"] = quote_compare["dif_rel"].abs() >= SUSPECT_THRESHOLD

suspeitas = (
    quote_compare[quote_compare["suspeita"]]
    .sort_values("dif_rel", key=lambda s: s.abs(), ascending=False)
    .reset_index(drop=True)
)

print(f"linhas comparadas: {len(quote_compare)}")
print(f"linhas suspeitas: {len(suspeitas)}")

display(
    suspeitas[
        [
            "papel",
            "Data",
            "Estratégia",
            "Volume Mínimo",
            "Ativos na Carteira",
            "cotacao_carteira",
            "cotacao_fundamentus",
            "dif_abs",
            "dif_rel",
            "Quantidade",
            "Valor",
        ]
    ].head(30)
)

linhas comparadas: 2730
linhas suspeitas: 538


,papel,Data,Estratégia,Volume Mínimo,Ativos na Carteira,cotacao_carteira,cotacao_fundamentus,dif_abs,dif_rel,Quantidade,Valor
0,AZUL4,2026-03-11,ROIC,1000000,30,3.75,0.81,2.94,3.629630,1778.0,6667.50
1,AZUL4,2026-03-10,ROIC,400000,30,3.75,0.81,2.94,3.629630,889.0,3333.75
2,AZUL4,2026-03-11,ROIC,400000,30,3.75,0.81,2.94,3.629630,1778.0,6667.50
3,AZUL4,2026-03-10,ROIC,1000000,30,3.75,0.81,2.94,3.629630,889.0,3333.75
4,MDNE3,2026-03-10,Earnings Yield,1000000,20,10.53,31.80,-21.27,-0.668868,475.0,5001.75
5,MDNE3,2026-03-11,Earnings Yield,1000000,20,10.53,31.80,-21.27,-0.668868,950.0,10003.50
6,MDNE3,2026-03-10,Earnings Yield,400000,20,10.53,31.80,-21.27,-0.668868,475.0,5001.75
7,MDNE3,2026-03-11,Earnings Yield,400000,30,10.53,31.80,-21.27,-0.668868,634.0,6676.02
8,MDNE3,2026-03-11,Earnings Yield,1000000,30,10.53,31.80,-21.27,-0.668868,634.0,6676.02
9,MDNE3,2026-03-10,Earnings Yield,1000000,30,10.53,31.80,-21.27,-0.668868,317.0,3338.01


In [4]:
resumo_suspeitas = (
    suspeitas.groupby(["papel", "Data"], as_index=False)
    .agg(
        ocorrencias=("papel", "size"),
        cotacao_carteira=("cotacao_carteira", "first"),
        cotacao_fundamentus=("cotacao_fundamentus", "first"),
        max_dif_rel=("dif_rel", lambda s: s.abs().max()),
    )
    .sort_values("max_dif_rel", ascending=False)
)

display(resumo_suspeitas.head(20))

,papel,Data,ocorrencias,cotacao_carteira,cotacao_fundamentus,max_dif_rel
9,AZUL4,2026-03-11,2,3.75,0.81,3.629630
8,AZUL4,2026-03-10,2,3.75,0.81,3.629630
41,MDNE3,2026-03-11,4,10.53,31.80,0.668868
40,MDNE3,2026-03-10,4,10.53,31.80,0.668868
5,ANIM3,2026-03-11,4,1.53,4.49,0.659243
4,ANIM3,2026-03-10,4,1.53,4.49,0.659243
60,SEER3,2026-03-10,4,4.27,11.24,0.620107
61,SEER3,2026-03-11,4,4.27,11.24,0.620107
20,CSMG3,2026-03-10,1,20.80,53.14,0.608581
21,CSMG3,2026-03-11,1,20.80,53.14,0.608581


## 2. Carteira vencedora recalculada com cotacoes do Fundamentus

A celula abaixo seleciona uma carteira vencedora. Se houver empate no retorno, o primeiro registro apos a ordenacao sera usado.

In [5]:
FREQUENCIA = "mensal"

winner = (
    analysis[analysis["frequencia"] == FREQUENCIA]
    .sort_values(["retorno_periodo", "Estratégia", "Volume Mínimo", "Ativos na Carteira"], ascending=[False, True, True, True])
    .iloc[0]
)

winner_frame = pd.DataFrame([winner])[
    [
        "frequencia",
        "periodo",
        "Estratégia",
        "Volume Mínimo",
        "Ativos na Carteira",
        "data_inicial",
        "data_final",
        "valor_inicial",
        "valor_final",
        "retorno_periodo",
    ]
]

display(winner_frame)

,frequencia,periodo,Estratégia,Volume Mínimo,Ativos na Carteira,data_inicial,data_final,valor_inicial,valor_final,retorno_periodo
40,mensal,2026-03-31,ROIC,400000,5,2026-03-10,2026-03-18,99991.23,495704.09,3.957476


In [6]:
def carteira_na_data(df, estrategia, volume_minimo, ativos_na_carteira, data):
    return (
        df[
            (df["Estratégia"] == estrategia)
            & (df["Volume Mínimo"] == volume_minimo)
            & (df["Ativos na Carteira"] == ativos_na_carteira)
            & (df["Data"] == pd.Timestamp(data))
        ]
        .copy()
        .reset_index(drop=True)
    )


def juntar_cotacao_fundamentus(df, fund_df):
    return df.merge(
        fund_df[["papel", "update date", "Cotação"]].rename(
            columns={"update date": "Data", "Cotação": "cotacao_fundamentus"}
        ),
        on=["papel", "Data"],
        how="left",
    )


def preparar_snapshot(snapshot, data_label):
    renamed = snapshot[["papel", "Data", "Quantidade", "Valor", "Cotação", "cotacao_fundamentus"]].copy()
    renamed = renamed.rename(
        columns={
            "Data": f"data_{data_label}",
            "Quantidade": f"quantidade_{data_label}",
            "Valor": f"valor_{data_label}_ativo_historico",
            "Cotação": f"cotacao_{data_label}_historico",
            "cotacao_fundamentus": f"cotacao_{data_label}",
        }
    )
    return renamed


In [7]:
carteira_inicial = carteira_na_data(
    latest_history,
    winner["Estratégia"],
    winner["Volume Mínimo"],
    winner["Ativos na Carteira"],
    winner["data_inicial"],
)
carteira_final = carteira_na_data(
    latest_history,
    winner["Estratégia"],
    winner["Volume Mínimo"],
    winner["Ativos na Carteira"],
    winner["data_final"],
)

carteira_inicial = juntar_cotacao_fundamentus(carteira_inicial, latest_fund)
carteira_final = juntar_cotacao_fundamentus(carteira_final, latest_fund)

inicial = preparar_snapshot(carteira_inicial, "inicial")
final = preparar_snapshot(carteira_final, "final")

comparativo = inicial.merge(final, on="papel", how="outer")

comparativo["status"] = np.select(
    [
        comparativo["quantidade_inicial"].isna(),
        comparativo["quantidade_final"].isna(),
    ],
    ["entrou", "saiu"],
    default="permaneceu",
)

comparativo["quantidade_inicial"] = comparativo["quantidade_inicial"].fillna(0)
comparativo["quantidade_final"] = comparativo["quantidade_final"].fillna(0)

comparativo["valor_inicial_ativo_recalculado"] = comparativo["quantidade_inicial"] * comparativo["cotacao_inicial"]
comparativo["valor_final_ativo_recalculado"] = comparativo["quantidade_final"] * comparativo["cotacao_final"]

total_inicial_recalculado = comparativo["valor_inicial_ativo_recalculado"].sum(min_count=1)
total_final_recalculado = comparativo["valor_final_ativo_recalculado"].sum(min_count=1)

comparativo["peso_inicial_recalculado"] = comparativo["valor_inicial_ativo_recalculado"] / total_inicial_recalculado
comparativo["peso_final_recalculado"] = comparativo["valor_final_ativo_recalculado"] / total_final_recalculado

comparativo["variacao_cotacao_fundamentus"] = np.where(
    comparativo["cotacao_inicial"].notna() & comparativo["cotacao_final"].notna() & comparativo["cotacao_inicial"].ne(0),
    comparativo["cotacao_final"] / comparativo["cotacao_inicial"] - 1,
    np.nan,
)

comparativo["variacao_valor_recalculado"] = np.where(
    comparativo["valor_inicial_ativo_recalculado"].notna() & comparativo["valor_inicial_ativo_recalculado"].ne(0),
    comparativo["valor_final_ativo_recalculado"] / comparativo["valor_inicial_ativo_recalculado"] - 1,
    np.nan,
)

comparativo["variacao_peso_recalculado"] = (
    comparativo["peso_final_recalculado"] - comparativo["peso_inicial_recalculado"]
)

comparativo["desvio_cotacao_inicial_vs_historico"] = np.where(
    comparativo["cotacao_inicial_historico"].notna() & comparativo["cotacao_inicial"].notna() & comparativo["cotacao_inicial"].ne(0),
    comparativo["cotacao_inicial_historico"] / comparativo["cotacao_inicial"] - 1,
    np.nan,
)
comparativo["desvio_cotacao_final_vs_historico"] = np.where(
    comparativo["cotacao_final_historico"].notna() & comparativo["cotacao_final"].notna() & comparativo["cotacao_final"].ne(0),
    comparativo["cotacao_final_historico"] / comparativo["cotacao_final"] - 1,
    np.nan,
)

comparativo = comparativo.sort_values(
    ["status", "valor_final_ativo_recalculado", "valor_inicial_ativo_recalculado"],
    ascending=[True, False, False],
).reset_index(drop=True)

resumo_recalculado = pd.DataFrame(
    [
        {
            "frequencia": winner["frequencia"],
            "periodo": winner["periodo"],
            "estrategia": winner["Estratégia"],
            "volume_minimo": winner["Volume Mínimo"],
            "ativos_na_carteira": winner["Ativos na Carteira"],
            "data_inicial": winner["data_inicial"],
            "data_final": winner["data_final"],
            "valor_inicial_historico": winner["valor_inicial"],
            "valor_final_historico": winner["valor_final"],
            "retorno_historico": winner["retorno_periodo"],
            "valor_inicial_recalculado": total_inicial_recalculado,
            "valor_final_recalculado": total_final_recalculado,
            "retorno_recalculado": total_final_recalculado / total_inicial_recalculado - 1,
        }
    ]
)

display(resumo_recalculado)
display(comparativo)

,frequencia,periodo,estrategia,volume_minimo,ativos_na_carteira,data_inicial,data_final,valor_inicial_historico,valor_final_historico,retorno_historico,valor_inicial_recalculado,valor_final_recalculado,retorno_recalculado
0,mensal,2026-03-31,ROIC,400000,5,2026-03-10,2026-03-18,99991.23,495704.09,3.957476,154475.86,495704.09,2.208942


,papel,data_inicial,quantidade_inicial,valor_inicial_ativo_historico,cotacao_inicial_historico,cotacao_inicial,data_final,quantidade_final,valor_final_ativo_historico,cotacao_final_historico,cotacao_final,status,valor_inicial_ativo_recalculado,valor_final_ativo_recalculado,peso_inicial_recalculado,peso_final_recalculado,variacao_cotacao_fundamentus,variacao_valor_recalculado,variacao_peso_recalculado,desvio_cotacao_inicial_vs_historico,desvio_cotacao_final_vs_historico
0,CURY3,2026-03-10,1156.0,19998.80,17.30,35.59,2026-03-18,3971.0,144147.30,36.30,36.30,permaneceu,41142.04,144147.30,0.266333,0.290793,0.019949,2.503650,0.024460,-0.513908,0.0
1,PLPL3,2026-03-10,2222.0,19998.00,9.00,14.14,2026-03-18,8691.0,125845.68,14.48,14.48,permaneceu,31419.08,125845.68,0.203392,0.253873,0.024045,3.005390,0.050481,-0.363508,0.0
2,PSSA3,2026-03-10,575.0,19992.75,34.77,49.96,2026-03-18,2382.0,114336.00,48.00,48.00,permaneceu,28727.00,114336.00,0.185964,0.230654,-0.039231,2.980088,0.044689,-0.304043,0.0
3,ODPV3,2026-03-10,1938.0,20000.16,10.32,13.71,2026-03-18,8393.0,111375.11,13.27,13.27,permaneceu,26569.98,111375.11,0.172001,0.224681,-0.032093,3.191765,0.052680,-0.247265,0.0
4,LEVE3,2026-03-10,764.0,20001.52,26.18,34.84,2026-03-18,0.0,NaN,34.49,34.49,saiu,26617.76,0.00,0.172310,0.000000,-0.010046,-1.000000,-0.172310,-0.248565,0.0


In [8]:
comparativo_formatado = comparativo.copy()

for col in [
    "peso_inicial_recalculado",
    "peso_final_recalculado",
    "variacao_cotacao_fundamentus",
    "variacao_valor_recalculado",
    "variacao_peso_recalculado",
    "desvio_cotacao_inicial_vs_historico",
    "desvio_cotacao_final_vs_historico",
]:
    comparativo_formatado[col] = comparativo_formatado[col].map(
        lambda x: f"{x:.2%}" if pd.notna(x) else "-"
    )

display(
    comparativo_formatado[
        [
            "papel",
            "status",
            "cotacao_inicial_historico",
            "cotacao_inicial",
            "cotacao_final_historico",
            "cotacao_final",
            "variacao_cotacao_fundamentus",
            "quantidade_inicial",
            "quantidade_final",
            "valor_inicial_ativo_recalculado",
            "valor_final_ativo_recalculado",
            "peso_inicial_recalculado",
            "peso_final_recalculado",
            "variacao_peso_recalculado",
            "desvio_cotacao_inicial_vs_historico",
            "desvio_cotacao_final_vs_historico",
        ]
    ]
)

,papel,status,cotacao_inicial_historico,cotacao_inicial,cotacao_final_historico,cotacao_final,variacao_cotacao_fundamentus,quantidade_inicial,quantidade_final,valor_inicial_ativo_recalculado,valor_final_ativo_recalculado,peso_inicial_recalculado,peso_final_recalculado,variacao_peso_recalculado,desvio_cotacao_inicial_vs_historico,desvio_cotacao_final_vs_historico
0,CURY3,permaneceu,17.30,35.59,36.30,36.30,1.99%,1156.0,3971.0,41142.04,144147.30,26.63%,29.08%,2.45%,-51.39%,0.00%
1,PLPL3,permaneceu,9.00,14.14,14.48,14.48,2.40%,2222.0,8691.0,31419.08,125845.68,20.34%,25.39%,5.05%,-36.35%,0.00%
2,PSSA3,permaneceu,34.77,49.96,48.00,48.00,-3.92%,575.0,2382.0,28727.00,114336.00,18.60%,23.07%,4.47%,-30.40%,0.00%
3,ODPV3,permaneceu,10.32,13.71,13.27,13.27,-3.21%,1938.0,8393.0,26569.98,111375.11,17.20%,22.47%,5.27%,-24.73%,0.00%
4,LEVE3,saiu,26.18,34.84,34.49,34.49,-1.00%,764.0,0.0,26617.76,0.00,17.23%,0.00%,-17.23%,-24.86%,0.00%
